# Stufe 5 — Chunking, Index und die Frage, ob es etwas taugt

**KI-gestützte Dokumentenaufbereitung · Referenz-Implementierung · Niveau DQR 5/6**

Stufe 2 hat aus jeder Seite einen `SeitenBefund` mit gefülltem Feld `text` gemacht.
Dieses Notebook baut daraus einen durchsuchbaren Index — und **misst**, was der
Aufwand der Stufen 1 und 2 im Retrieval tatsächlich einbringt.

Die Reihenfolge ist Absicht. Die naheliegende Fortsetzung wäre gewesen, erst mit
einem Sprachmodell anzureichern und dann zu indexieren. Dann wüsste man aber nicht,
ob die Anreicherung nötig war. Umgekehrt zeigt der Index, **wo** er blind ist, und
erst diese Lücken rechtfertigen ein weiteres Modell.

## Einordnung ins Phasenmodell

| Phase | Inhalt | in diesem Notebook |
|---|---|---|
| 01b, 01c | Layout und Erkennung | erledigt — wird eingelesen |
| 02 | Datenmodellierung | Chunk-Schema |
| **05** | **Verankerung** | **Schwerpunkt** — jede Fundstelle mit Seite und Rechteck |
| 06 | Evaluation | Recall@k und MRR gegen Prüffragen |
| 04b | Anreicherung mit LLM | wird hier **begründet**, nicht gebaut |

## Was hier zum ersten Mal zusammenkommt

Eine gewöhnliche `PDF → Text → alle 800 Zeichen schneiden`-Pipeline hat vier Dinge
nicht, die wir aus den Stufen 1 und 2 mitbringen:

| aus dem Befund | wofür im Index |
|---|---|
| `pp_label` | an Elementgrenzen schneiden statt mitten im Satz |
| `lese_index` | richtige Reihenfolge, auch bei zwei Spalten |
| `strom` | Kopf- und Fußzeilen kommen gar nicht erst hinein |
| `bbox` + `seite` | jede Fundstelle zeigt auf ein Rechteck im PDF |

Der letzte Punkt ist die **Verankerung** aus dem Leitfaden. Eine Antwort, die auf
Seite 112, Block 7, Rechteck (833, 1409, 1442, 1620) verweist, ist nachprüfbar. Die
meisten Pipelines können das nicht, weil sie die Geometrie schon beim Extrahieren
wegwerfen.

```bash
pip install requests numpy pydantic
```

In [1]:
from __future__ import annotations

import json
import re
import time
from enum import Enum
from pathlib import Path

import numpy as np
import requests
from pydantic import BaseModel, Field

LMS_URL      = "http://localhost:1234/v1"
EINBETTUNG   = "text-embedding-multilingual-e5-large-instruct"   # ANPASSEN
BEFUND_DIR   = Path("befunde")
INDEX_DATEI  = Path("index/chunks.json")

# Das Zeitfenster des Einbettungsmodells, nicht Geschmackssache - siehe Abschnitt 3.
ZIEL_ZEICHEN = 1200
MAX_ZEICHEN  = 1500

INDEX_DATEI.parent.mkdir(exist_ok=True)

BEFUNDE = {}
for pfad in sorted(BEFUND_DIR.glob("*_stufe2.json")):
    name = pfad.stem.replace("_stufe2", "")
    BEFUNDE[name] = json.loads(pfad.read_text(encoding="utf-8"))
    b = BEFUNDE[name]
    mit_text = sum(1 for blk in b["bloecke"] if blk.get("text"))
    print(f"{name:14s} {len(b['bloecke']):3d} Blöcke, {mit_text:3d} mit Text")

assert BEFUNDE, "Keine *_stufe2.json gefunden. Erst Notebook 3 laufen lassen."

tietze          22 Blöcke,  21 mit Text
zimbardo        11 Blöcke,  11 mit Text


---
## 1. Der Überschriftenpfad

Ein Chunk aus der Mitte eines Kapitels ist ohne Kontext schwer zu treffen. Steht in
ihm „Der benötigte Zuwachs ist immer ein Zehntel der Standardlänge", fehlt jeder
Hinweis auf *Weber* und *Psychophysik* — die Wörter, mit denen jemand danach sucht.

Diese Information liegt aber vor. `doc_title` und `paragraph_title` in
Lesereihenfolge ergeben einen Baum, und jeder Block darunter erbt seinen Pfad.

Die Umsetzung ist ein Stapel: ein `doc_title` leert ihn und wird neue Wurzel, ein
`paragraph_title` ersetzt die unterste Ebene. Das ist bewusst flach — PP-DocLayoutV3
kennt keine Überschriftenebenen, nur *Titel* und *Absatztitel*. Eine tiefere
Hierarchie ließe sich aus der Nummerierung („4.2.2") ableiten, das wäre aber schon
Interpretation und gehört in Stufe 4.

In [2]:
BOILERPLATE = {"header", "footer", "header_image", "footer_image", "number"}
APPARAT = {"footnote", "vision_footnote", "reference", "reference_content"}

def strom_fuer(pp_label: str) -> str:
    if pp_label in BOILERPLATE:  return "boilerplate"
    if pp_label == "aside_text": return "marginalie"
    if pp_label in APPARAT:      return "apparat"
    return "haupt"


def sortiert(bloecke: list[dict]) -> list[dict]:
    """Blöcke in Lesereihenfolge - lese_index, ersatzweise id."""
    return sorted(bloecke, key=lambda b: (b.get("lese_index") if b.get("lese_index")
                                          is not None else 10**6, b["id"]))


def ueberschriftenpfade(bloecke: list[dict]) -> dict[int, list[str]]:
    """Block-id -> Liste der übergeordneten Überschriften, von außen nach innen."""
    pfade: dict[int, list[str]] = {}
    stapel: list[str] = []
    for blk in sortiert(bloecke):
        text = (blk.get("text") or "").strip()
        if blk["pp_label"] == "doc_title" and text:
            stapel = [text]
        elif blk["pp_label"] == "paragraph_title" and text:
            stapel = stapel[:1] + [text]
        pfade[blk["id"]] = list(stapel)
    return pfade


for name, befund in BEFUNDE.items():
    pfade = ueberschriftenpfade(befund["bloecke"])
    tiefste = max(pfade.values(), key=len, default=[])
    print(f"{name:14s} tiefster Pfad: {' > '.join(tiefste) or '(keiner)'}")

tietze         tiefster Pfad: (keiner)
zimbardo       tiefster Pfad: 4.2.2 Von physikalischen zu mentalen Ereignissen


---
## 2. Chunk-Schema

Ein Chunk ist **nicht** nur Text. Er trägt alles mit, was nötig ist, um von der
Fundstelle zurück ins PDF zu kommen.

`bbox` ist die Vereinigung der beteiligten Blockrechtecke. Das ist geometrisch grob —
bei zwei Absätzen untereinander umschließt sie auch den Zwischenraum — aber für den
Zweck genau richtig: es geht um „zeig mir, wo das steht", nicht um pixelgenaue
Markierung. Wer die genauer braucht, hat über `block_ids` den Weg zurück in den
Befund.

In [3]:
class Chunk(BaseModel):
    id: str = Field(description="quelle:seite:laufnummer")
    quelle_datei: str
    seite: int
    block_ids: list[int]
    strom: str
    text_format: str = "klartext"
    ueberschriften: list[str] = Field(default_factory=list)
    text: str
    bbox: tuple[float, float, float, float]
    render_dpi: int

    @property
    def kontext(self) -> str:
        """Text mit vorangestelltem Überschriftenpfad - das, was eingebettet wird."""
        if not self.ueberschriften:
            return self.text
        return " > ".join(self.ueberschriften) + "\n\n" + self.text

    def zitat(self) -> str:
        x0, y0, x1, y1 = self.bbox
        return (f"{self.quelle_datei}, S. {self.seite + 1}, "
                f"Rechteck ({x0:.0f}, {y0:.0f}, {x1:.0f}, {y1:.0f}) "
                f"bei {self.render_dpi} dpi")

---
## 3. Die Chunk-Größe ist keine Geschmacksfrage

Verbreitete Faustregeln nennen 500 oder 1000 Zeichen mit etwas Überlappung. Beides
ist geraten. Die tatsächliche Schranke steht im Modell:

**Die multilingual-E5-Modelle verarbeiten höchstens 512 Tokens. Alles darüber wird
stillschweigend abgeschnitten.** Ohne Fehlermeldung, ohne Warnung — der Chunk ist im
Index, aber seine zweite Hälfte ist unsichtbar.

Für deutschen Text mit dem XLM-RoBERTa-Tokenizer liegt man bei grob drei bis vier
Zeichen je Token. 512 Tokens sind also etwa 1500 bis 2000 Zeichen. Der Zielwert von
1200 lässt Luft für den Überschriftenpfad, der ja mit eingebettet wird.

> ⚠️ „Stillschweigend" ist das Gefährliche daran. Ein zu langer Chunk **funktioniert**
> — er liefert eine Einbettung, er landet im Index, er wird manchmal gefunden. Nur
> ist der hintere Teil nie durchsuchbar. Deshalb prüft die Zelle unten die Länge und
> meldet Überschreitungen, statt sie hinzunehmen.

### Die Regeln

| Blocktyp | Behandlung |
|---|---|
| `boilerplate` | **verworfen** — kommt gar nicht in den Index |
| Bild ohne Text | verworfen, aber vermerkt (Lücke für Stufe 4b) |
| Haupttext | zusammengefasst bis `ZIEL_ZEICHEN`, Grenze immer an einer Blockgrenze |
| `table`, `chart` | **eigener Chunk**, nie mit Fließtext vermischt |
| Überschrift | beginnt einen neuen Chunk und führt ihn an |
| Chunk unter 120 Zeichen | an den Vorgänger gehängt, falls Strom und Format passen |
| Marginalie, Apparat | eigene Chunks, mit `strom` markiert |

Die Trennung von Tabellen ist wichtiger, als sie aussieht. Eine Einbettung mittelt
über alles im Chunk; OTSL-Marken und Fließtext in einem Vektor ergeben einen, der
weder das eine noch das andere gut trifft.

Und es gibt **keine Überlappung** zwischen Chunks. Der übliche Grund dafür — der Text
könnte mitten im Gedanken zerschnitten sein — entfällt, weil hier an Elementgrenzen
geschnitten wird und nicht nach Zeichenzahl. Das ist der unmittelbarste Gewinn aus
Stufe 1.

In [4]:
EIGENER_CHUNK = {"table", "chart", "display_formula", "algorithm", "seal"}
UEBERSCHRIFT = {"doc_title", "paragraph_title"}
MIN_ZEICHEN = 120


def chunken(name: str, befund: dict) -> tuple[list[Chunk], list[str]]:
    pfade = ueberschriftenpfade(befund["bloecke"])
    quelle = befund["quelle_datei"]
    seite = befund["seite"]
    dpi = befund["render_dpi"]

    chunks: list[Chunk] = []
    luecken: list[str] = []
    puffer: list[dict] = []

    def vereinige(bloecke: list[dict]) -> tuple[float, float, float, float]:
        return (min(b["bbox"]["x0"] for b in bloecke), min(b["bbox"]["y0"] for b in bloecke),
                max(b["bbox"]["x1"] for b in bloecke), max(b["bbox"]["y1"] for b in bloecke))

    def abschliessen(format_: str = "klartext") -> None:
        nonlocal puffer
        if not puffer:
            return
        chunks.append(Chunk(
            id="", quelle_datei=quelle, seite=seite,
            block_ids=[b["id"] for b in puffer],
            strom=strom_fuer(puffer[0]["pp_label"]),
            text_format=format_,
            ueberschriften=pfade[puffer[0]["id"]],
            text="\n\n".join((b.get("text") or "").strip() for b in puffer),
            bbox=vereinige(puffer), render_dpi=dpi))
        puffer = []

    letzter_strom = None
    for blk in sortiert(befund["bloecke"]):
        strom = strom_fuer(blk["pp_label"])
        text = (blk.get("text") or "").strip()

        if strom == "boilerplate":
            continue
        if not text:
            luecken.append(f"#{blk['id']} {blk['pp_label']}"
                           + (f" -> {blk['ausschnitt']}" if blk.get("ausschnitt") else ""))
            continue

        eigener = blk["pp_label"] in EIGENER_CHUNK
        # Eine Überschrift beginnt einen neuen Chunk und führt ihn an.
        ueberschrift = blk["pp_label"] in UEBERSCHRIFT
        wechsel = strom != letzter_strom
        zu_lang = sum(len(b.get("text") or "") for b in puffer) + len(text) > ZIEL_ZEICHEN

        if eigener or ueberschrift or wechsel or zu_lang:
            abschliessen()
        letzter_strom = strom

        puffer.append(blk)
        if eigener:
            abschliessen(blk.get("text_format") or "klartext")

    abschliessen()
    return nummerieren(verschmelzen(chunks), name), luecken


def verschmelzen(chunks: list[Chunk]) -> list[Chunk]:
    """Sehr kurze Fließtext-Chunks an den Vorgänger hängen.

    Ein Chunk aus zwanzig Zeichen ist im Index Rauschen: sein Vektor wird von
    wenigen Tokens bestimmt und trifft zufällig. Zusammengelegt wird nur mit einem
    Nachbarn desselben Stroms und Formats - eine Tabelle bleibt eine Tabelle.
    """
    ergebnis: list[Chunk] = []
    for c in chunks:
        vor = ergebnis[-1] if ergebnis else None
        if (vor is not None and len(c.text) < MIN_ZEICHEN
                and c.text_format == "klartext" and vor.text_format == "klartext"
                and c.strom == vor.strom
                and len(vor.text) + len(c.text) <= ZIEL_ZEICHEN):
            vor.text += "\n\n" + c.text
            vor.block_ids += c.block_ids
            vor.bbox = (min(vor.bbox[0], c.bbox[0]), min(vor.bbox[1], c.bbox[1]),
                        max(vor.bbox[2], c.bbox[2]), max(vor.bbox[3], c.bbox[3]))
            continue
        ergebnis.append(c)
    return ergebnis


def nummerieren(chunks: list[Chunk], name: str) -> list[Chunk]:
    for i, c in enumerate(chunks):
        c.id = f"{name}:{c.seite}:{i:03d}"
    return chunks


CHUNKS: list[Chunk] = []
LUECKEN: dict[str, list[str]] = {}
for name, befund in BEFUNDE.items():
    teile, luecken = chunken(name, befund)
    CHUNKS += teile
    LUECKEN[name] = luecken
    print(f"{name:14s} {len(teile):3d} Chunks, {len(luecken)} ohne Text")

zu_lang = [c for c in CHUNKS if len(c.kontext) > MAX_ZEICHEN]
zu_kurz = [c for c in CHUNKS if len(c.text) < MIN_ZEICHEN]
print(f"\n{len(CHUNKS)} Chunks gesamt, "
      f"Median {int(np.median([len(c.kontext) for c in CHUNKS]))} Zeichen, "
      f"längster {max(len(c.kontext) for c in CHUNKS)}")
if zu_lang:
    print(f"! {len(zu_lang)} über {MAX_ZEICHEN} Zeichen - Gefahr stiller Abschneidung:")
    for c in zu_lang:
        print(f"    {c.id} {len(c.kontext)} Zeichen, Blöcke {c.block_ids}")
if zu_kurz:
    print(f"! {len(zu_kurz)} unter {MIN_ZEICHEN} Zeichen - isoliert, kein Nachbar passte:")
    for c in zu_kurz:
        print(f"    {c.id} [{c.text_format}] {c.text[:50]!r}")

print("\nLücken (Blöcke ohne Text - Kandidaten für Stufe 4b):")
for name, l in LUECKEN.items():
    for eintrag in l:
        print(f"  {name}: {eintrag}")

tietze           6 Chunks, 1 ohne Text
zimbardo         7 Chunks, 0 ohne Text

13 Chunks gesamt, Median 298 Zeichen, längster 1101
! 4 unter 120 Zeichen - isoliert, kein Nachbar passte:
    tietze:0:001 [latex] '$$\n|\\underline{{i}}_{r,R}|^{2}=\\frac{4k T}{R}\n$$'
    tietze:0:003 [latex] '$$\n|\\underline{{u}}_{r,T}|^{2}=\\frac{2k T U_{T}}{I'
    tietze:0:004 [latex] '$$\n|\\underline{{i}}_{r,T}|^{2}=\\frac{2q I_{C,A}}{\\'
    zimbardo:0:001 [klartext] '$L_{a}-L_{b}=\\Delta L$'

Lücken (Blöcke ohne Text - Kandidaten für Stufe 4b):
  tietze: #2 image -> ausschnitte/tietze_02_image.png


---
## 4. Einbetten — und die häufigste E5-Falle

E5-Modelle sind mit **Rollenpräfixen** trainiert. Lässt man sie weg, sinkt die
Qualität, und zwar genau dort, wo es in einer Demo nicht auffällt.

Dabei gibt es zwei unvereinbare Schemata, und die Verwechslung ist der zweithäufigste
E5-Fehler überhaupt:

| Variante | Dokumente | Anfragen |
|---|---|---|
| `multilingual-e5-large` (ohne `-instruct`) | `passage: …` | `query: …` |
| `multilingual-e5-large-**instruct**` | **kein Präfix** | `Instruct: {Aufgabe}\nQuery: {Anfrage}` |

Wir nutzen die Instruct-Variante, also: Chunks ohne Präfix, Anfragen mit
Aufgabenbeschreibung. Die Funktion unten macht den Unterschied durch zwei getrennte
Aufrufwege explizit, damit er nicht versehentlich verlorengeht.

> 💡 Falls du ein anderes Modell einsetzt: `EINBETTUNG` oben ändern **und** die
> Präfixe hier anpassen. Das ist eine der wenigen Stellen im Projekt, an denen zwei
> Einstellungen zwingend zusammenpassen müssen.

In [5]:
AUFGABE = ("Given a question about a textbook page, "
           "retrieve the passage that answers it")


def _einbetten(texte: list[str], stapel: int = 16) -> np.ndarray:
    vektoren = []
    for i in range(0, len(texte), stapel):
        antwort = requests.post(f"{LMS_URL}/embeddings",
                                json={"model": EINBETTUNG, "input": texte[i:i + stapel]},
                                timeout=300)
        antwort.raise_for_status()
        daten = sorted(antwort.json()["data"], key=lambda d: d["index"])
        vektoren += [d["embedding"] for d in daten]
    v = np.asarray(vektoren, dtype=np.float32)
    return v / np.clip(np.linalg.norm(v, axis=1, keepdims=True), 1e-9, None)


def dokumente_einbetten(texte: list[str]) -> np.ndarray:
    """Instruct-Variante: Dokumente bekommen KEIN Präfix."""
    return _einbetten(texte)


def anfrage_einbetten(frage: str) -> np.ndarray:
    """Instruct-Variante: Anfragen bekommen Aufgabenbeschreibung plus Query."""
    return _einbetten([f"Instruct: {AUFGABE}\nQuery: {frage}"])[0]


t0 = time.perf_counter()
MATRIX = dokumente_einbetten([c.kontext for c in CHUNKS])
print(f"{MATRIX.shape[0]} Chunks eingebettet, Dimension {MATRIX.shape[1]}, "
      f"{time.perf_counter() - t0:.1f} s")
print(f"Normiert: {np.allclose(np.linalg.norm(MATRIX, axis=1), 1.0)}")

13 Chunks eingebettet, Dimension 1024, 0.4 s
Normiert: True


---
## 5. Suche mit Verankerung

Weil alle Vektoren auf Länge 1 normiert sind, ist die Kosinus-Ähnlichkeit ein
einfaches Skalarprodukt. Bei 30 Chunks ist eine Vektordatenbank überflüssig — und es
schadet nicht, einmal gesehen zu haben, dass darin nichts Geheimnisvolles steckt.

Ausgegeben wird nicht nur der Text, sondern die **Fundstelle**: Datei, Seite,
Rechteck. Das ist der Unterschied zwischen „das Modell behauptet etwas" und „das
steht hier, sieh selbst nach".

In [6]:
def suchen(frage: str, k: int = 5) -> list[tuple[Chunk, float]]:
    v = anfrage_einbetten(frage)
    werte = MATRIX @ v
    beste = np.argsort(-werte)[:k]
    return [(CHUNKS[i], float(werte[i])) for i in beste]


def zeigen(frage: str, k: int = 3) -> None:
    print(f"? {frage}\n")
    for rang, (c, wert) in enumerate(suchen(frage, k), 1):
        pfad = " > ".join(c.ueberschriften) or "(ohne Überschrift)"
        print(f"{rang}. {wert:.3f}  [{c.strom}/{c.text_format}]  {pfad}")
        print(f"   {c.zitat()}")
        print(f"   {c.text[:200].replace(chr(10), ' ')}\n")


zeigen("Wie groß ist die Weber'sche Konstante für Schallfrequenz?")

? Wie groß ist die Weber'sche Konstante für Schallfrequenz?

1. 0.905  [haupt/otsl]  (ohne Überschrift)
   Zimbardo_Psychologie_18Aufl_Extracted.pdf, S. 1, Rechteck (238, 1593, 763, 1959) bei 200 dpi
   <fcel>Reizdimension<fcel>Weber’sche Konstante (k)<nl><fcel>Schallfrequenz<fcel>0,003<nl><fcel>Lichtintensität<fcel>0,01<nl><fcel>Geruchs-konzentration<fcel>0,07<nl><fcel>Druckintensität<fcel>0,14<nl><

2. 0.844  [haupt/klartext]  (ohne Überschrift)
   Zimbardo_Psychologie_18Aufl_Extracted.pdf, S. 1, Rechteck (199, 1139, 1444, 1565) bei 200 dpi
   Abbildung 4.8: Eben merkliche Unterschiede und Weber‘sches Gesetz. Angenommen, Sie führen ein Experiment durch, dessen Teilnehmer angeben sollen, ob zwei Striche gleich oder verschieden lang sind. Je 

3. 0.837  [haupt/klartext]  (ohne Überschrift)
   HalbleiterSchaltungstechnik_TietzeSchenk_2002_Extracted.pdf, S. 1, Rechteck (89, 945, 1058, 1111) bei 200 dpi
   Damit wird allerdings nur das thermische Rauschen eines idealen Widerstands erfasst

---
## 6. Messen statt schätzen

Eine Suche, die bei drei Beispielen plausibel aussieht, sagt nichts. Also ein kleines
Prüfset: Frage plus eine **Zeichenkette, die im richtigen Chunk vorkommen muss**.

Der Trick mit der Zeichenkette statt einer Chunk-Nummer ist Absicht. Chunk-Nummern
ändern sich, sobald man an den Regeln aus Abschnitt 3 dreht — und genau das will man
ja tun. Ein inhaltlicher Marker überlebt das.

Zwei Maße:

- **Recall@k** — wie oft ist der richtige Chunk unter den ersten k. Das ist die
  Frage, ob die Antwort überhaupt im Kontext eines nachgeschalteten Modells landet.
- **MRR** (Mean Reciprocal Rank) — der Kehrwert des Rangs, gemittelt. Rang 1 gibt
  1,0, Rang 2 gibt 0,5, Rang 4 gibt 0,25. Das misst, ob der richtige Chunk auch
  **oben** steht, nicht nur dabei ist.

> ⚠️ Sechs Fragen sind kein Benchmark. Sie reichen, um grobe Fehler zu finden und um
> zwei Varianten gegeneinander zu halten — nicht, um eine Zahl zu veröffentlichen.
> Die Fragen unten sind aus dem Seiteninhalt abgeleitet; ergänzen Sie eigene,
> besonders solche, die Sie für schwierig halten.

In [7]:
PRUEFFRAGEN = [
    {"quelle": "zimbardo",
     "frage": "Wie groß ist die Weber'sche Konstante für Schallfrequenz?",
     "erwartet": "0,003"},
    {"quelle": "zimbardo",
     "frage": "Warum braucht eine Getränkefabrik viel Zucker für eine merklich süßere Cola?",
     "erwartet": "Getränkefabrik"},
    {"quelle": "zimbardo",
     "frage": "Wie verhält sich der nötige Längenzuwachs zur Länge des Referenzstrichs?",
     "erwartet": "Zehntel"},
    {"quelle": "tietze",
     "frage": "Was muss ein rauscharmer Bipolartransistor aufweisen?",
     "erwartet": "Basisbahnwiderstand"},
    {"quelle": "tietze",
     "frage": "Welche Bauelemente rauschen in integrierten Schaltungen nicht?",
     "erwartet": "rauschfrei"},
    {"quelle": "tietze",
     "frage": "Wie kann man die Rauschspannungsdichte durch die Bauform beeinflussen?",
     "erwartet": "Skalierung"},
]

# Nur Fragen zu Seiten, die tatsächlich im Index sind.
PRUEFFRAGEN = [f for f in PRUEFFRAGEN if f["quelle"] in BEFUNDE]
print(f"{len(PRUEFFRAGEN)} Prüffragen für {sorted(BEFUNDE)}")


def rang_von(frage: str, erwartet: str, matrix: np.ndarray,
             chunks: list[Chunk], k: int = 10) -> int | None:
    """Rang des ersten Chunks, der den Marker enthält. None = nicht in den ersten k."""
    v = anfrage_einbetten(frage)
    for rang, i in enumerate(np.argsort(-(matrix @ v))[:k], 1):
        if erwartet.lower() in chunks[i].text.lower():
            return rang
    return None


def auswerten(matrix: np.ndarray, chunks: list[Chunk],
              beschriftung: str, zeige_details: bool = True) -> dict:
    raenge = []
    for f in PRUEFFRAGEN:
        r = rang_von(f["frage"], f["erwartet"], matrix, chunks)
        raenge.append(r)
        if zeige_details:
            marke = f"Rang {r}" if r else "nicht gefunden"
            print(f"  {marke:16s} {f['frage'][:62]}")

    n = len(raenge)
    ergebnis = {
        "variante": beschriftung,
        "recall@1": sum(1 for r in raenge if r == 1) / n,
        "recall@3": sum(1 for r in raenge if r and r <= 3) / n,
        "recall@5": sum(1 for r in raenge if r and r <= 5) / n,
        "mrr": sum(1 / r for r in raenge if r) / n,
    }
    print(f"\n{beschriftung}:  R@1 {ergebnis['recall@1']:.2f}   "
          f"R@3 {ergebnis['recall@3']:.2f}   R@5 {ergebnis['recall@5']:.2f}   "
          f"MRR {ergebnis['mrr']:.3f}")
    return ergebnis


BASIS = auswerten(MATRIX, CHUNKS, "mit Überschriftenkontext")

6 Prüffragen für ['tietze', 'zimbardo']
  Rang 1           Wie groß ist die Weber'sche Konstante für Schallfrequenz?
  Rang 1           Warum braucht eine Getränkefabrik viel Zucker für eine merklic
  Rang 2           Wie verhält sich der nötige Längenzuwachs zur Länge des Refere
  Rang 1           Was muss ein rauscharmer Bipolartransistor aufweisen?
  Rang 1           Welche Bauelemente rauschen in integrierten Schaltungen nicht?
  Rang 1           Wie kann man die Rauschspannungsdichte durch die Bauform beein

mit Überschriftenkontext:  R@1 0.83   R@3 1.00   R@5 1.00   MRR 0.917


---
## 7. Experiment A — bringt der Überschriftenpfad etwas?

Bisher ist das eine Behauptung: der vorangestellte Pfad soll Chunks auffindbar
machen, denen der Themenbezug im eigenen Text fehlt. Prüfen wir es.

Dieselben Chunks, dieselben Fragen, einmal eingebettet **mit** und einmer **ohne**
Pfad. Alles andere bleibt gleich — das ist der Punkt eines A/B-Vergleichs.

Erwartung: der Unterschied ist bei Fragen groß, deren Suchbegriffe im Chunk selbst
nicht vorkommen, und null bei Fragen, deren Antwort ohnehin wörtlich im Text steht.
Falls **kein** Unterschied herauskommt, ist entweder das Prüfset zu leicht oder die
Überschriften tragen auf diesen Seiten nichts bei.

In [8]:
MATRIX_OHNE = dokumente_einbetten([c.text for c in CHUNKS])
OHNE = auswerten(MATRIX_OHNE, CHUNKS, "ohne Überschriftenkontext")

print("\n" + "=" * 62)
print(f"{'Variante':30s} {'R@1':>6} {'R@3':>6} {'R@5':>6} {'MRR':>7}")
for e in (BASIS, OHNE):
    print(f"{e['variante']:30s} {e['recall@1']:6.2f} {e['recall@3']:6.2f} "
          f"{e['recall@5']:6.2f} {e['mrr']:7.3f}")

  Rang 1           Wie groß ist die Weber'sche Konstante für Schallfrequenz?
  Rang 1           Warum braucht eine Getränkefabrik viel Zucker für eine merklic
  Rang 2           Wie verhält sich der nötige Längenzuwachs zur Länge des Refere
  Rang 1           Was muss ein rauscharmer Bipolartransistor aufweisen?
  Rang 1           Welche Bauelemente rauschen in integrierten Schaltungen nicht?
  Rang 1           Wie kann man die Rauschspannungsdichte durch die Bauform beein

ohne Überschriftenkontext:  R@1 0.83   R@3 1.00   R@5 1.00   MRR 0.917

Variante                          R@1    R@3    R@5     MRR
mit Überschriftenkontext         0.83   1.00   1.00   0.917
ohne Überschriftenkontext        0.83   1.00   1.00   0.917


---
## 8. Experiment B — die Tabellen

Hier erwarte ich, dass der Index **scheitert**, und das ist der Zweck des
Experiments.

OTSL ist hervorragend, um eine Tabelle zu rekonstruieren, und untauglich, um sie zu
finden. Der Chunk enthält `<fcel>Schallfrequenz<fcel>0,003<nl>` — die Zellinhalte
stehen da, aber eingebettet wird ein Vektor über einen Text, der zu einem guten Teil
aus Strukturmarken besteht. Eine Frage in natürlicher Sprache liegt in diesem Raum
weit weg.

Die Zelle unten misst den Abstand zwischen beiden Welten direkt: dieselbe Frage gegen
den OTSL-Chunk und gegen eine von Hand formulierte Zusammenfassung desselben Inhalts.

In [9]:
otsl_chunks = [c for c in CHUNKS if c.text_format == "otsl"]
print(f"{len(otsl_chunks)} OTSL-Chunks im Index\n")

for c in otsl_chunks:
    print(f"{c.id}  Blöcke {c.block_ids}  {len(c.text)} Zeichen")
    print(f"  {c.text[:120]}...\n")

if otsl_chunks:
    frage = "Wie groß ist die Weber'sche Konstante für Schallfrequenz?"
    v = anfrage_einbetten(frage)

    # Wo landet der OTSL-Chunk in der Rangliste?
    ordnung = list(np.argsort(-(MATRIX @ v)))
    for c in otsl_chunks:
        i = CHUNKS.index(c)
        print(f"{c.id}: Rang {ordnung.index(i) + 1} von {len(CHUNKS)}, "
              f"Ähnlichkeit {float(MATRIX[i] @ v):.3f}")

    # Gegenprobe: derselbe Inhalt in Prosa. Von Hand - genau das würde Stufe 4b tun.
    prosa = ("Tabelle der Weber'schen Konstanten für ausgewählte Reizdimensionen. "
             "Schallfrequenz 0,003, Lichtintensität 0,01, Geruchskonzentration 0,07, "
             "Druckintensität 0,14, Schallintensität 0,15, Geschmackskonzentration 0,20.")
    v_prosa = dokumente_einbetten([prosa])[0]
    print(f"\nDieselbe Tabelle als Prosa: Ähnlichkeit {float(v_prosa @ v):.3f}")
    print("Ist dieser Wert deutlich höher, ist Stufe 4b belegt - nicht vermutet.")

2 OTSL-Chunks im Index

zimbardo:0:000  Blöcke [1]  305 Zeichen
  <fcel>A. Eben merklich längere Striche<fcel>B. Standard-strichlänge<fcel>L_{a} - L_{b} = \(\Delta L\)<nl><fcel>11 mm<fce...

zimbardo:0:004  Blöcke [6]  279 Zeichen
  <fcel>Reizdimension<fcel>Weber’sche Konstante (k)<nl><fcel>Schallfrequenz<fcel>0,003<nl><fcel>Lichtintensität<fcel>0,01<...

zimbardo:0:000: Rang 5 von 13, Ähnlichkeit 0.816
zimbardo:0:004: Rang 1 von 13, Ähnlichkeit 0.905

Dieselbe Tabelle als Prosa: Ähnlichkeit 0.911
Ist dieser Wert deutlich höher, ist Stufe 4b belegt - nicht vermutet.


---
## 9. Index ablegen

In [10]:
INDEX_DATEI.write_text(
    json.dumps({
        "einbettungsmodell": EINBETTUNG,
        "aufgabe": AUFGABE,
        "ziel_zeichen": ZIEL_ZEICHEN,
        "chunks": [c.model_dump(mode="json") for c in CHUNKS],
    }, ensure_ascii=False, indent=2), encoding="utf-8")

np.save(INDEX_DATEI.with_suffix(".npy"), MATRIX)
print(f"{INDEX_DATEI}  {INDEX_DATEI.stat().st_size / 1024:.1f} kB")
print(f"{INDEX_DATEI.with_suffix('.npy')}  {MATRIX.nbytes / 1024:.1f} kB")

index/chunks.json  10.6 kB
index/chunks.npy  52.0 kB


---
## 10. Was die Messung beantwortet — und was nicht

Nach diesem Notebook liegt fest:

| Frage | Antwort steht in |
|---|---|
| Findet der Index Fließtext zuverlässig? | Abschnitt 6, Recall@3 und MRR |
| Trägt der Überschriftenpfad? | Abschnitt 7, Vergleich der beiden Zeilen |
| Sind Tabellen auffindbar? | Abschnitt 8, Rang des OTSL-Chunks |
| Wie viele Blöcke haben gar keinen Text? | Abschnitt 3, Liste der Lücken |

Die letzten beiden Zeilen sind die **Begründung für Stufe 4b**. Wenn der OTSL-Chunk
auf Rang 12 von 30 landet, die Prosafassung desselben Inhalts aber auf Rang 1, dann
ist eine LLM-Zusammenfassung je Tabelle kein Feature-Wunsch, sondern eine belegte
Lücke. Dasselbe für Bildblöcke, die überhaupt keinen Text haben und im Index schlicht
nicht existieren.

Das war der Grund, Stufe 5 vor Stufe 4b zu bauen.

## Offene Punkte

| # | Punkt | Wie zu klären |
|---|---|---|
| 1 | Sechs Prüffragen sind zu wenig | auf 30 bis 50 erweitern, davon die Hälfte bewusst schwierig |
| 2 | Der Marker-Trick prüft Anwesenheit, nicht Relevanz | ein Chunk, der `0,003` zufällig enthält, zählt als Treffer |
| 3 | Nur eine Seite je Dokument | bei mehreren hundert Chunks ändern sich die Rangfolgen; Vollsuche wird dann langsam |
| 4 | Kein Vergleich mit naivem Chunking | die Behauptung „besser als alle 800 Zeichen schneiden" ist noch ungeprüft |
| 5 | Keine Volltextsuche als Rückfall | reine Vektorsuche versagt bei Eigennamen und Zahlen; hybrid mit BM25 ist der übliche Ausweg |
| 6 | Zeichen als Näherung für Tokens | die 512er-Grenze ist in Tokens; wer sie ausreizen will, muss tokenisieren statt zählen |

Punkt 4 ist der lohnendste. Das ganze Projekt behauptet, dass Layout-Analyse dem
Retrieval nützt — gemessen ist das noch nicht. Der Vergleich wäre einfach: denselben
Seitentext einmal ohne jede Struktur alle 800 Zeichen schneiden, einbetten, dieselben
Prüffragen. Das ist Übungsaufgabe C.

---

## Übungsaufgaben

**A — Die Grenze finden (Einstieg).**
Setzen Sie `ZIEL_ZEICHEN` auf 4000 und lassen Sie Abschnitt 6 erneut laufen. Die
Chunks sind dann länger als das Modellfenster. Fällt das Ergebnis ab? Erklären Sie,
warum der Abfall **kleiner** ausfällt, als man erwarten würde — und warum das die
Fehlersuche erschwert.

**B — Präfixe verwechseln (mittel).**
Ändern Sie `dokumente_einbetten` so, dass es `passage: ` voranstellt — das Schema der
Nicht-Instruct-Variante. Messen Sie erneut. Notieren Sie den Unterschied und
begründen Sie, warum dieser Fehler in einer Vorführung mit fünf Beispielen fast nie
auffällt.

**C — Gegen naives Chunking antreten (Projektaufgabe).**
Bauen Sie einen zweiten Index: `page.get_text()` je Seite, alle 800 Zeichen
geschnitten, 100 Zeichen Überlappung, keine Metadaten. Dieselben Prüffragen.
Vergleichen Sie R@1, R@3 und MRR. Achten Sie besonders auf die Tietze-Seite — dort
ist der Textlayer die Lüge aus Notebook 1.

**D — Verankerung sichtbar machen (fortgeschritten).**
Schreiben Sie eine Funktion, die zu einem Suchergebnis die PDF-Seite rendert und das
`bbox` des Chunks einzeichnet. Diskutieren Sie, was die vereinigte Bbox über mehrere
Blöcke ungenau macht und wann man stattdessen die Einzelrechtecke aus `block_ids`
nachschlagen sollte.

**E — Hybrid (Projektaufgabe).**
Ergänzen Sie eine BM25-Suche über dieselben Chunks und mischen Sie die Ränge. An
welchen der sechs Prüffragen gewinnt die Volltextsuche, an welchen die Vektorsuche?
Was sagt das über die Fragen, die Ihre Schüler:innen später wirklich stellen werden?